In [1]:

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import zipfile
import os

In [3]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

print("Device:", device)

Device: cuda


In [18]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    ])

In [17]:
import zipfile
with zipfile.ZipFile('/content/Rock-Paper-Scissors.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

In [7]:
print(os.listdir('/content/Rock-Paper-Scissors/test'))

['paper', 'scissors', 'rock']


In [19]:
test_data = datasets.ImageFolder(
    root='/content/Rock-Paper-Scissors/test',
    transform=transform
)

train_data = datasets.ImageFolder(
    root='/content/Rock-Paper-Scissors/train',
    transform=transform
)

In [20]:
test_loader = DataLoader(
    test_data,
    batch_size=32,
    shuffle=True
)

train_loader = DataLoader(
    train_data,
    batch_size=32,
)

In [34]:
images, labels = next(iter(train_loader))
images = images.numpy()

In [35]:
class RockPaperScissors(nn.Module):
    def __init__(self):
        super(RockPaperScissors, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size= 3, stride= 1, padding= 1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size= 2, stride= 2)

        self.conv2 = nn.Conv2d(16, 32, kernel_size= 3, stride= 1, padding= 1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size= 2, stride= 2)

        self.conv3 = nn.Conv2d(32, 64, kernel_size= 3, stride= 1, padding= 1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size= 2, stride= 2)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 16 * 16, 128)
        self.relu4 = nn.ReLU()
        self.fc2 = nn.Linear(128, 3)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        x = self.conv3(x)
        x = self.relu3(x)
        x = self.pool3(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu4(x)
        x = self.fc2(x)
        return x








In [36]:
model = RockPaperScissors().to(device)

In [37]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [38]:
epochs = 15

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")


Epoch 1, Loss: 7.3335
Epoch 2, Loss: 1.1351
Epoch 3, Loss: 1.0667
Epoch 4, Loss: 1.4364
Epoch 5, Loss: 1.1237
Epoch 6, Loss: 1.1058
Epoch 7, Loss: 1.1058
Epoch 8, Loss: 1.1058
Epoch 9, Loss: 1.1057
Epoch 10, Loss: 1.1057
Epoch 11, Loss: 1.1056
Epoch 12, Loss: 1.1055
Epoch 13, Loss: 1.1055
Epoch 14, Loss: 1.1055
Epoch 15, Loss: 1.1055


In [39]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 33.33%


In [43]:
torch.save(model.state_dict(), 'model.pth')